In [1]:
#download_data
import kagglehub
import shutil
import os

path = kagglehub.dataset_download("nudratabbas/global-ads-performance-google-meta-tiktok")
print("Downloaded to cache:", path)

dest = "data"
os.makedirs(dest, exist_ok=True)
for f in os.listdir(path):
    shutil.copy(os.path.join(path, f), os.path.join(dest, f))
    print("Copied:", f)

100%|███████████████████| 58.8k/58.8k [00:00<00:00, 2.05MB/s]

Extracting files...
Downloaded to cache: C:\Users\HP\.cache\kagglehub\datasets\nudratabbas\global-ads-performance-google-meta-tiktok\versions\1
Copied: global_ads_performance_dataset.csv


In [2]:
#audit_data
import pandas as pd
import glob

files = glob.glob("data/*.csv")
print("Files found:", files)

for f in files:
    df = pd.read_csv(f)
    print(f"\n{'='*60}\nFILE: {f}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns & dtypes:\n{df.dtypes}")
    print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"\nFirst 3 rows:\n{df.head(3)}")
    
    date_cols = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()]
    for dc in date_cols:
        print(f"\nDate range [{dc}]: {df[dc].min()} -> {df[dc].max()}")
    
    channel_cols = [c for c in df.columns if 'platform' in c.lower() or 'channel' in c.lower()]
    for cc in channel_cols:
        print(f"\nUnique [{cc}]: {df[cc].unique()}")
    
    print(f"\nSummary stats:\n{df.describe()}")

Files found: ['data\\global_ads_performance_dataset.csv']

FILE: data\global_ads_performance_dataset.csv
Shape: (1800, 14)

Columns & dtypes:
date              object
platform          object
campaign_type     object
industry          object
country           object
impressions        int64
clicks             int64
CTR              float64
CPC              float64
ad_spend         float64
conversions        int64
CPA              float64
revenue          float64
ROAS             float64
dtype: object

Missing values:
Series([], dtype: int64)

First 3 rows:
         date    platform campaign_type    industry country  impressions  \
0  2024-01-21  Google Ads        Search     Fintech     UAE        59886   
1  2024-01-22  TikTok Ads        Search      EdTech      UK       135608   
2  2024-06-15  TikTok Ads         Video  Healthcare     USA        92313   

   clicks     CTR   CPC  ad_spend  conversions    CPA   revenue   ROAS  
0    2113  0.0353  1.26   2662.38          159  16.74   480

In [1]:
#Exploratory breakdown
import pandas as pd
pd.set_option('display.width', 120)

df = pd.read_csv("data/global_ads_performance_dataset.csv")
df['date'] = pd.to_datetime(df['date'])

print("=== CATEGORICAL DIMENSIONS ===")
for col in ['platform', 'campaign_type', 'industry', 'country']:
    print(f"\n{col} ({df[col].nunique()} unique):")
    print(df[col].value_counts())

print("\n=== GRANULARITY CHECK ===")
print("Total rows:", len(df))
print("Unique dates:", df['date'].nunique())
print("Rows per (date, platform):")
print(df.groupby(['date','platform']).size().value_counts())  

print("\n=== PLATFORM-LEVEL SUMMARY ===")
platform_summary = df.groupby('platform').agg(
    total_spend=('ad_spend','sum'),
    total_revenue=('revenue','sum'),
    avg_ROAS=('ROAS','mean'),
    median_ROAS=('ROAS','median'),
    avg_CPA=('CPA','mean'),
    n_records=('platform','size')
).round(2)
platform_summary['blended_ROAS'] = (platform_summary['total_revenue']/platform_summary['total_spend']).round(2)
platform_summary['spend_share_%'] = (platform_summary['total_spend']/platform_summary['total_spend'].sum()*100).round(1)
platform_summary['revenue_share_%'] = (platform_summary['total_revenue']/platform_summary['total_revenue'].sum()*100).round(1)
print(platform_summary)

print("\n=== CAMPAIGN_TYPE x PLATFORM: mean ROAS ===")
print(df.pivot_table(index='campaign_type', columns='platform', values='ROAS', aggfunc='mean').round(2))

print("\n=== SPEND vs REVENUE CORRELATION per platform ===")
for p in df['platform'].unique():
    sub = df[df['platform']==p]
    print(f"{p}: corr(spend, revenue) = {sub['ad_spend'].corr(sub['revenue']):.3f}, corr(spend, ROAS) = {sub['ad_spend'].corr(sub['ROAS']):.3f}")

print("\n=== MONTHLY TREND (blended ROAS) ===")
df['month'] = df['date'].dt.to_period('M')
print(df.groupby(['month','platform'])['ROAS'].mean().unstack().round(2))

=== CATEGORICAL DIMENSIONS ===

platform (3 unique):
platform
Google Ads    720
Meta Ads      630
TikTok Ads    450
Name: count, dtype: int64

campaign_type (4 unique):
campaign_type
Search      477
Video       456
Shopping    447
Display     420
Name: count, dtype: int64

industry (5 unique):
industry
EdTech        372
SaaS          370
Fintech       361
E-commerce    349
Healthcare    348
Name: count, dtype: int64

country (7 unique):
country
UK           266
USA          266
Canada       262
India        261
UAE          258
Germany      255
Australia    232
Name: count, dtype: int64

=== GRANULARITY CHECK ===
Total rows: 1800
Unique dates: 360
Rows per (date, platform):
1    341
2    293
3    132
4     68
5     30
6      8
7      1
Name: count, dtype: int64

=== PLATFORM-LEVEL SUMMARY ===
            total_spend  total_revenue  avg_ROAS  median_ROAS  avg_CPA  n_records  blended_ROAS  spend_share_%  \
platform                                                                          